# Data analysis, session 2B: One reusable stock report
**October 15, 10:35-11:15 (40 minutes).** This notebook runs independently and also serves as the Papermill template.

- 10 minutes: parameters, filtered SQL, and validation.
- 15 minutes: adjusted returns, close-to-close rolling volatility, drawdown, and a summary.
- 10 minutes: assemble a 2x2 figure and export results.
- 5 minutes: change the ticker; explain warm-up rows and units.

The live calculation uses the same rolling standard deviation introduced on October 8 and needed for Homework 2.
Yang-Zhang OHLC volatility remains an executable appendix after the main report.
Run with the project Python kernel; the bCourses database must be in this lecture's `data/` folder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures/Lecture 4'
plt.style.use('seaborn-v0_8-whitegrid')
import scrapbook as sb
import matplotlib.dates as mdates

In [ ]:
ticker = 'AAPL'
start_date = '2024-01-01'
end_date = '2024-06-30'
db_path = str(lesson_dir / 'data/data.db')
volatility_window = 20

## 1. Load Data from Database

In [ ]:
database = Path(db_path).expanduser().resolve()
if not database.is_file():
    raise FileNotFoundError(f'Download the bCourses database to {database}')
if pd.Timestamp(start_date) > pd.Timestamp(end_date):
    raise ValueError('start_date must be on or before end_date.')
if not isinstance(volatility_window, int) or isinstance(volatility_window, bool) or volatility_window < 2:
    raise ValueError('volatility_window must be an integer >= 2.')
query = """
SELECT ts, open, high, low, close, adj_close, volume
FROM ohlc WHERE ticker = ? AND ts >= ? AND ts <= ? ORDER BY ts
"""
from contextlib import closing
with closing(sqlite3.connect(database.as_uri() + '?mode=ro', uri=True)) as conn:
    df = pd.read_sql_query(query, conn, params=(ticker, start_date, end_date), parse_dates=['ts'], index_col='ts')
if len(df) < volatility_window + 1:
    raise ValueError(f'{ticker}: need at least {volatility_window + 1} prices for a full return window; got {len(df)}.')
print(f'Loaded {len(df)} trading days for {ticker}: {df.index.min().date()} to {df.index.max().date()}')
display(df.head())

## 2. Validate before calculating
The same contract from module 2A is repeated here so the template is self-contained.
Missing values, infinity, nonpositive prices, duplicate dates, or invalid OHLC/volume cause a clear failure rather than a misleading report.
This validates observed rows; it does not certify full calendar coverage or vendor accuracy.

In [ ]:
def quality_flags(frame):
    """Flag missing/non-finite values, nonpositive equity prices, bad OHLC, and duplicate dates."""
    price_cols = ['open', 'high', 'low', 'close', 'adj_close']
    flags = pd.DataFrame(index=frame.index)
    flags['missing_or_nonfinite'] = ~np.isfinite(frame[price_cols + ['volume']]).all(axis=1)
    flags['nonpositive_price'] = frame[price_cols].le(0).any(axis=1)
    flags['ohlc_bounds'] = (frame['high'].lt(frame[['open', 'close']].max(axis=1))
                            | frame['low'].gt(frame[['open', 'close']].min(axis=1))
                            | frame['high'].lt(frame['low']))
    flags['negative_volume'] = frame['volume'].lt(0)
    flags['duplicate_date'] = frame.index.duplicated(keep=False)
    return flags

quality = quality_flags(df)
display(quality.sum())
data_quality_ok = bool(not quality.any().any())
if not data_quality_ok:
    raise ValueError('Data quality checks failed: ' + str(quality.sum().to_dict()))

## 3. Calculate features
Adjusted close accounts for the vendor's adjustment convention; raw OHLC remains useful for within-day checks.
Daily return is a fraction; volatility is annualized with 252 trading days. The first return and the rolling warm-up region are undefined.
Point-in-time drawdown is price/running peak minus one; its minimum is maximum drawdown.
See the [pandas return documentation](https://pandas.pydata.org/docs/reference/api/pandas.Series.pct_change.html) for fractional-change semantics.

In [ ]:
df['daily_return'] = df['adj_close'].pct_change(fill_method=None)
df['log_return'] = np.log(df['adj_close'] / df['adj_close'].shift(1))
df['cumulative_return'] = df['adj_close'] / df['adj_close'].iloc[0] - 1
df['drawdown'] = df['adj_close'] / df['adj_close'].cummax() - 1

In [ ]:
df['volatility'] = df['daily_return'].rolling(volatility_window).std() * np.sqrt(252)
assert df['volatility'].iloc[:volatility_window].isna().all()
assert np.isfinite(df['volatility'].iloc[volatility_window:]).all()
print(f'Calculated {volatility_window}-day close-to-close volatility')

In [ ]:
# Intraday range as percentage of price
df['daily_range_pct'] = (df['high'] - df['low']) / df['close'] * 100

# Simple moving averages
df['sma_20'] = df['adj_close'].rolling(20).mean()
df['sma_50'] = df['adj_close'].rolling(50).mean()

# Volume moving average (for relative volume)
df['volume_sma_20'] = df['volume'].rolling(20).mean()
df['relative_volume'] = df['volume'] / df['volume_sma_20']

## 4. Summary Statistics

In [ ]:
# Calculate key statistics
stats = {
    'ticker': ticker,
    'start_date': start_date,
    'end_date': end_date,
    'trading_days': len(df),
    'start_price': df['adj_close'].iloc[0],
    'end_price': df['adj_close'].iloc[-1],
    'total_return_pct': df['cumulative_return'].iloc[-1] * 100,
    'avg_daily_return_pct': df['daily_return'].mean() * 100,
    'daily_return_std_pct': df['daily_return'].std() * 100,
    'avg_volatility': df['volatility'].mean(),
    'max_volatility': df['volatility'].max(),
    'min_volatility': df['volatility'].min(),
    'max_drawdown_pct': df['drawdown'].min() * 100,
    'avg_daily_volume': df['volume'].mean(),
    'max_single_day_return_pct': df['daily_return'].max() * 100,
    'min_single_day_return_pct': df['daily_return'].min() * 100,
}

# Display stats
print(f"\n=== {ticker} Summary: {start_date} to {end_date} ===")
print(f"Trading days: {stats['trading_days']}")
print(f"Price: ${stats['start_price']:.2f} -> ${stats['end_price']:.2f}")
print(f"Total return: {stats['total_return_pct']:.2f}%")
print(f"Average daily return: {stats['avg_daily_return_pct']:.4f}%")
print(f"Daily return std dev: {stats['daily_return_std_pct']:.4f}%")
print(f"Average volatility (annualized): {stats['avg_volatility']:.2%}")
print(f"Max drawdown: {stats['max_drawdown_pct']:.2f}%")
print(f"Best single day: {stats['max_single_day_return_pct']:.2f}%")
print(f"Worst single day: {stats['min_single_day_return_pct']:.2f}%")

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'{ticker} Analysis: {start_date} to {end_date}', fontsize=14, fontweight='bold')

# Plot 1: Price with moving averages
ax1 = axes[0, 0]
ax1.plot(df.index, df['adj_close'], label='Adjusted Close', linewidth=1.5)
ax1.plot(df.index, df['sma_20'], label='20-day SMA', linewidth=1, alpha=0.7)
ax1.plot(df.index, df['sma_50'], label='50-day SMA', linewidth=1, alpha=0.7)
ax1.set_ylabel('Price ($)')
ax1.set_title('Price with Moving Averages')
ax1.legend(loc='upper left', fontsize=8)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 2: Volume
ax2 = axes[0, 1]
ax2.bar(df.index, df['volume'] / 1e6, width=1, alpha=0.7, label='Daily Volume')
ax2.plot(df.index, df['volume_sma_20'] / 1e6, color='red', linewidth=1, label='20-day Avg')
ax2.set_ylabel('Volume (millions)')
ax2.set_title('Trading Volume')
ax2.legend(loc='upper right', fontsize=8)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 3: Rolling Volatility
ax3 = axes[1, 0]
ax3.plot(df.index, df['volatility'] * 100, linewidth=1.5, color='purple')
ax3.axhline(y=df['volatility'].mean() * 100, color='gray', linestyle='--', 
            label=f'Mean: {df["volatility"].mean():.1%}')
ax3.set_ylabel('Volatility (%)')
ax3.set_title(f'{volatility_window}-Day Rolling Volatility (Annualized)')
ax3.legend(loc='upper right', fontsize=8)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Plot 4: Daily Returns Distribution
ax4 = axes[1, 1]
returns_clean = df['daily_return'].dropna()
ax4.hist(returns_clean * 100, bins=50, edgecolor='black', alpha=0.7)
ax4.axvline(x=returns_clean.mean() * 100, color='red', linestyle='--', 
            label=f'Mean: {returns_clean.mean()*100:.3f}%')
ax4.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax4.set_xlabel('Daily Return (%)')
ax4.set_ylabel('Frequency')
ax4.set_title('Daily Returns Distribution')
ax4.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Export Results with Scrapbook

We use `scrapbook` to save key outputs so they can be collected when running many notebooks at scale.

In [ ]:
# Glue individual metrics (these can be easily aggregated later)
sb.glue('ticker', ticker)
sb.glue('start_date', start_date)
sb.glue('end_date', end_date)
sb.glue('trading_days', stats['trading_days'])
sb.glue('total_return_pct', stats['total_return_pct'])
sb.glue('avg_volatility', stats['avg_volatility'])
sb.glue('max_drawdown_pct', stats['max_drawdown_pct'])
sb.glue('data_quality_ok', data_quality_ok)

# Glue the full stats dictionary for detailed access
serializable_stats = {key: value.item() if isinstance(value, np.generic) else value for key, value in stats.items()}
sb.glue('stats', serializable_stats)

# Glue the processed dataframe for potential further analysis
sb.glue('prices', df, encoder='pandas')

# Glue the figure
sb.glue('analysis_plot', fig, encoder='display', display=False)

print("Results exported via scrapbook.")
print("These can be collected when running this notebook at scale with papermill.")

## Checkpoint: change the ticker (5 minutes)
Try `MSFT`, then Restart & Run All. Predict which outputs will change.
Why do the first `volatility_window` rows lack a volatility estimate? Why is `adj_close` excluded from raw OHLC bounds checks?

Next, [run the report across stocks](3_running_analysis_at_scale.ipynb).
Papermill finds the cell tagged `parameters`, injects overrides after it, and saves an executed notebook.
Scrapbook stores the explicitly named metrics in that notebook's outputs.

## Optional appendix: Yang-Zhang volatility (outside live timing)
The original OHLC estimator is retained here for further study. It combines sample variances of overnight and open-to-close log returns with the mean Rogers-Satchell term.
All OHLC prices are multiplied by the same daily adjustment factor before calculating cross-day ratios.
This is an estimate from historical adjusted data, not a point-in-time trading signal.
Reference: [Yang and Zhang (2000)](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=229190).
The main report and batch comparison above use close-to-close volatility.

In [ ]:
def yang_zhang_volatility(frame, window=20):
    """Annualized Yang-Zhang estimate using sample variances and mean Rogers-Satchell term."""
    if window < 2:
        raise ValueError('window must be >= 2')
    factor = frame['adj_close'] / frame['close']
    adjusted = frame[['open', 'high', 'low', 'close']].mul(factor, axis=0)
    overnight = np.log(adjusted['open'] / adjusted['close'].shift(1))
    open_close = np.log(adjusted['close'] / adjusted['open'])
    high_open = np.log(adjusted['high'] / adjusted['open'])
    low_open = np.log(adjusted['low'] / adjusted['open'])
    rs = high_open * (high_open - open_close) + low_open * (low_open - open_close)
    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    variance = (overnight.rolling(window).var(ddof=1)
                + k * open_close.rolling(window).var(ddof=1)
                + (1 - k) * rs.rolling(window).mean())
    return np.sqrt(variance.clip(lower=0) * 252)

yz_vol = yang_zhang_volatility(df, volatility_window)
display(pd.DataFrame({'close_to_close': df['volatility'], 'yang_zhang': yz_vol}).tail())